In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import polars as pl
from influxdb_client  import InfluxDBClient , WriteOptions
import yaml
import requests


current_file = Path(globals().get('__vsc_ipynb_file__', None))
parent_dir = current_file.parent.parent
os.chdir(parent_dir)
print("Changed working directory to:", Path.cwd())

Changed working directory to: /home/legacy/Projects/Weather_forecasting_with_MLOps


In [3]:
from dotenv import load_dotenv
load_dotenv("mlops_services/.env.mlops")

True

In [4]:
os.getenv('INFLUXDB_URL')

'http://localhost:8086'

In [5]:
from mlops_services.src.utils.load_config import Config
weather_config = Config.from_yaml(Path('mlops_services/config/weather_data.yaml'))

In [6]:
weather_config.all_cities

['BLR', 'DEL']

In [ ]:
from mlops_services.src.components.weather_access import HistoricWeather
hist_weather = HistoricWeather(weather_config,'BLR')

In [8]:
hist_df = hist_weather.get_df( start='2023-01-01',end='2025-06-30')
hist_df.head()

time,temperature_2m,relative_humidity_2m,dew_point_2m,rain,weather_code,surface_pressure,cloud_cover,et0_fao_evapotranspiration,vapour_pressure_deficit,wind_speed_10m,wind_direction_10m,wind_gusts_10m,is_day,time_local,city,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation
"datetime[μs, UTC]",f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,"datetime[μs, Asia/Kolkata]",str,f64,f64,f64,i32,str,str,f64
2022-12-31 18:30:00 UTC,16.2,74,11.7,0.0,0,914.8,1,0.02,0.47,9.4,97,15.1,0,2023-01-01 00:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2022-12-31 19:30:00 UTC,16.1,75,11.6,0.0,0,914.1,0,0.02,0.46,9.8,96,16.2,0,2023-01-01 01:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2022-12-31 20:30:00 UTC,15.6,80,12.1,0.0,3,913.3,83,0.01,0.36,9.8,96,16.2,0,2023-01-01 02:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2022-12-31 21:30:00 UTC,15.3,84,12.7,0.0,2,913.0,78,0.0,0.27,10.2,100,16.9,0,2023-01-01 03:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2022-12-31 22:30:00 UTC,14.8,89,13.1,0.0,2,912.9,61,0.0,0.18,9.1,99,16.9,0,2023-01-01 04:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0


In [90]:
with open('units.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(hist_df['hourly_units'][0], f, allow_unicode=True)

In [74]:
res.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

In [11]:
hist_df.tail()


time,temperature_2m,relative_humidity_2m,dew_point_2m,rain,weather_code,surface_pressure,cloud_cover,et0_fao_evapotranspiration,vapour_pressure_deficit,wind_speed_10m,wind_direction_10m,wind_gusts_10m,is_day,time_local,city,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation
"datetime[μs, UTC]",f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,"datetime[μs, Asia/Kolkata]",str,f64,f64,f64,i32,str,str,f64
2025-06-30 13:30:00 UTC,23.9,72,18.5,0.1,51,908.4,100,0.07,0.84,11.7,268,34.2,0,2025-06-30 19:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2025-06-30 14:30:00 UTC,23.4,75,18.8,0.1,51,909.0,100,0.04,0.71,14.8,254,31.3,0,2025-06-30 20:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2025-06-30 15:30:00 UTC,22.6,78,18.7,0.0,3,909.4,99,0.04,0.59,15.9,260,36.4,0,2025-06-30 21:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2025-06-30 16:30:00 UTC,21.9,82,18.7,0.0,3,909.7,97,0.02,0.47,14.9,263,35.6,0,2025-06-30 22:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0
2025-06-30 17:30:00 UTC,21.3,85,18.7,0.0,3,909.6,100,0.01,0.38,13.1,262,33.1,0,2025-06-30 23:00:00 IST,"""BLR""",12.970123,77.56364,8.804321,19800,"""Asia/Kolkata""","""GMT+5:30""",918.0


In [13]:
from mlops_services.src.utils.db_io import InfluxDB
db = InfluxDB(bucket='WEATHER_DATA_RAW',measurement="city_BLR1")
db.ping()


InfluxDB connection successful.


True

In [10]:
db.delete_city()

In [12]:
db.push_to_db(hist_df)

In [9]:
from mlops_services.src.utils.time_utils import convert_to_utc


In [26]:
convert_to_utc('2023-01-01 00:00','2025-06-30 23:00', tz="Asia/Kolkata", from_format="%Y-%m-%d %H:%M",
                                                        to_format="%Y-%m-%dT%H:%M")

('2023-01-01 00:00', '2025-06-30 23:00')


('2022-12-31T18:30', '2025-06-30T17:30')

In [14]:
start , end = convert_to_utc('2023-01-01 00:00','2025-07-01 00:00', tz="Asia/Kolkata",from_format="%Y-%m-%d %H:%M",
                                                        to_format="%Y-%m-%dT%H:%M:%SZ")
hist_df1 = db.fetch_as_df(start=start,end=end)
hist_df1

('2023-01-01 00:00', '2025-07-01 00:00')


shape: (0, 0)
┌┐
╞╡
└┘

In [12]:
db.measurement

'city_BLR'

In [16]:
hist_df1.write_csv('mlops_services/data/historic/historic_BLR.csv')
hist_df1.write_parquet('mlops_services/data/historic/historic_BLR.parquet')

In [17]:
hist_df2 = pl.read_parquet('mlops_services/data/historic/historic_BLR.parquet')
hist_df2

result,table,_start,_stop,_time,_measurement,city,cloud_cover,dew_point_2m,elevation,et0_fao_evapotranspiration,generationtime_ms,is_day,latitude,longitude,rain,relative_humidity_2m,surface_pressure,temperature_2m,time_local,timezone,timezone_abbreviation,utc_offset_seconds,vapour_pressure_deficit,weather_code,wind_direction_10m,wind_gusts_10m,wind_speed_10m
str,i64,"datetime[ns, UTC]","datetime[ns, UTC]","datetime[ns, UTC]",str,str,i64,f64,f64,f64,f64,i64,f64,f64,f64,i64,f64,f64,str,str,str,i64,f64,i64,i64,f64,f64
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2022-12-31 18:30:00 UTC,"""city_BLR""","""BLR""",1,11.7,918.0,0.02,8.804321,0,12.970123,77.56364,0.0,74,914.8,16.2,"""2023-01-01 00:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.47,0,97,15.1,9.4
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2022-12-31 19:30:00 UTC,"""city_BLR""","""BLR""",0,11.6,918.0,0.02,8.804321,0,12.970123,77.56364,0.0,75,914.1,16.1,"""2023-01-01 01:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.46,0,96,16.2,9.8
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2022-12-31 20:30:00 UTC,"""city_BLR""","""BLR""",83,12.1,918.0,0.01,8.804321,0,12.970123,77.56364,0.0,80,913.3,15.6,"""2023-01-01 02:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.36,3,96,16.2,9.8
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2022-12-31 21:30:00 UTC,"""city_BLR""","""BLR""",78,12.7,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,84,913.0,15.3,"""2023-01-01 03:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.27,2,100,16.9,10.2
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2022-12-31 22:30:00 UTC,"""city_BLR""","""BLR""",61,13.1,918.0,0.0,8.804321,0,12.970123,77.56364,0.0,89,912.9,14.8,"""2023-01-01 04:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.18,2,99,16.9,9.1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2025-06-30 13:30:00 UTC,"""city_BLR""","""BLR""",100,18.5,918.0,0.07,8.804321,0,12.970123,77.56364,0.1,72,908.4,23.9,"""2025-06-30 19:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.84,51,268,34.2,11.7
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2025-06-30 14:30:00 UTC,"""city_BLR""","""BLR""",100,18.8,918.0,0.04,8.804321,0,12.970123,77.56364,0.1,75,909.0,23.4,"""2025-06-30 20:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.71,51,254,31.3,14.8
"""_result""",0,2022-12-31 18:30:00 UTC,2025-06-30 18:30:00 UTC,2025-06-30 15:30:00 UTC,"""city_BLR""","""BLR""",99,18.7,918.0,0.04,8.804321,0,12.970123,77.56364,0.0,78,909.4,22.6,"""2025-06-30 21:00:00+05:30""","""Asia/Kolkata""","""GMT+5:30""",19800,0.59,3,260,36.4,15.9


In [47]:
js=hist_weather.pull_data()

In [43]:
base_url = "https://archive-api.open-meteo.com/v1/archive"


response = requests.get(base_url, params=hist_weather.params)
response.json()



{'reason': "Invalid date format. Make sure to use 'YYYY-MM-DD'", 'error': True}